In [ ]:
# MOdule and import installation
!pip install -q langgraph langchain-openai tenacity langchain_groq

import os, json, time, operator, functools
from typing import TypedDict, Annotated, Literal, Optional
from collections import defaultdict
from langgraph.graph  import StateGraph, START, END
from langgraph.types  import Send
from langchain_core.messages import HumanMessage
from tenacity import retry, stop_after_attempt, wait_exponential
from pydantic import BaseModel
from langchain_groq import ChatGroq


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 99.6/99.6 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 3.3 MB/s eta 0:00:00


In [ ]:
# ── Measurement Infrastructure (FIRST, always) ────────────────────
node_latency  = defaultdict(list)    # node_name → [ms, ms, ...]
signal_counts = defaultdict(int)     # track fan-out signal quality
routing_dist  = defaultdict(int)     # track which paths are taken
llm = ChatGroq(model="llama-3.1-8b-instant", groq_api_key="")



In [ ]:
# Timed function & Agent State & Data set
def timed(name):
  def dec(fn):
    @functools.wraps(fn)
    def wrap(*a, **k):
      t = time.perf_counter()
      r = fn(*a, **k)
      node_latency[name].append((time.perf_counter() - t)*1000)
      return r
    return wrap


class TradeState(TypedDict):
  #Input
  trade_id: str
  trade_data: dict # {type, amount, currency}
  # Fan-In target - all 4 parallel analyst agents append here
  signals: Annotated[list, operator.add]
  # Set by ContextPruner
  active_signals: list
  # Set by ComplianceGuardrail
  compliance_passed: bool
  compliance_notes: list
  # Set by SettlementRouter
  urgency: Literal["CRITICAL", "STANDARD", "QUARANTINE"]
  decision: str
  # Immutable audit log
  audit_log: Annotated[list, operator.add]

  # ── Test trades covering all paths ───────────────────────────────
TEST_TRADES = [
    {"trade_id":"T001", "trade_data":{"type":"FX_SPOT", "amount":50_000_000, "currency":"USD/JPY", "counterparty":"MUFG", "settlement_type":"T+0"}},   # CRITICAL fast path
    {"trade_id":"T002", "trade_data":{"type":"EQUITY", "amount":2_500_000,  "currency":"GBP",     "counterparty":"BARC", "settlement_type":"T+2"}},   # Standard path
    {"trade_id":"T003", "trade_data":{"type":"DERIVATIVE", "amount":9_99_999_999,"currency":"EUR",     "counterparty":"UNKN", "settlement_type":"T+2"}},   # Compliance fail → quarantine
]


In [ ]:
# ── STEP 2: Ingest + Fast-Path Check + Parallel Agents ───────────

@timed("trade_ingest")
def trade_ingest(state: TradeState) -> dict:
    t = state["trade_data"]
    print(f"[Ingest] {state['trade_id']} | {t['type']} | ${t['amount']:,} | {t['settlement_type']}")
    return {"signals":[], "audit_log":[f"[INGEST] Trade {state['trade_id']} received"]}

def fast_path_router(state: TradeState):
    # T+0 settlement bypasses all analyst enrichment — speed is compliance here
    if state["trade_data"]["settlement_type"] == "T+0":
        routing_dist["fast_path"] += 1
        return "critical_path_node"
    routing_dist["standard_path"] += 1
    return [
        Send("risk_agent",      {**state}),
        Send("liquidity_agent", {**state}),
        Send("cparty_agent",    {**state}),
        Send("reg_agent",       {**state}),
    ]

def critical_path_node(state: TradeState) -> dict:
    # T+0: no LLM, no enrichment, immediate escalation packet
    print(f"  ⚡ [CRITICAL PATH] T+0 settlement — bypassing enrichment")
    signal = {"source":"fast_path", "ts":time.time(),
              "urgency":"CRITICAL", "requires_human":True}
    return {"signals":[signal], "active_signals":[signal],
            "compliance_passed":True, "compliance_notes":[],
            "audit_log":["[FAST PATH] T+0 — escalated without enrichment"]}

# ── Parallel Analyst Agents ───────────────────────────────────────
@timed("risk_agent")
@retry(stop=stop_after_attempt(3), wait=wait_exponential(min=1,max=4))
def risk_agent(state: TradeState) -> dict:
    t = state["trade_data"]
    risk_level = "HIGH" if t["amount"] > 10_000_000 else "MEDIUM"
    signal = {"source":"risk", "ts":time.time(),
              "risk_level":risk_level, "exposure_usd":t["amount"]}
    signal_counts["risk"] += 1
    print(f"  [RiskAgent] level={risk_level} exposure=${t['amount']:,}")
    return {"signals":[signal], "audit_log":["[RISK] Signal collected"]}

@timed("liquidity_agent")
def liquidity_agent(state: TradeState) -> dict:
    available = 75_000_000  # mock treasury balance
    t = state["trade_data"]
    coverage = available / t["amount"] if t["amount"] > 0 else 99
    signal = {"source":"liquidity", "ts":time.time(),
              "coverage_ratio":coverage, "adequate":coverage > 1.2}
    print(f"  [LiquidityAgent] coverage={coverage:.2f}x")
    return {"signals":[signal], "audit_log":["[LIQ] Signal collected"]}

@timed("cparty_agent")
def cparty_agent(state: TradeState) -> dict:
    APPROVED = ["MUFG","BARC","CITI","HSBC","JPM","GS"]
    cp = state["trade_data"]["counterparty"]
    signal = {"source":"counterparty", "ts":time.time(),
              "approved": cp in APPROVED, "counterparty":cp}
    print(f"  [CPartyAgent] {cp} → {'✓ approved' if signal['approved'] else '✗ not in approved list'}")
    return {"signals":[signal], "audit_log":[f"[CPARTY] {cp} checked"]}

@timed("reg_agent")
def reg_agent(state: TradeState) -> dict:
    t = state["trade_data"]
    # MiFID II: derivatives > $500M require pre-trade reporting
    needs_pretrade = t["type"] == "DERIVATIVE" and t["amount"] > 500_000_000
    signal = {"source":"regulatory", "ts":time.time(),
              "mifid_pretrade_required":needs_pretrade,
              "reporting_breach":needs_pretrade}
    print(f"  [RegAgent] MiFID pre-trade required={needs_pretrade}")
    return {"signals":[signal], "audit_log":["[REG] Regulatory check complete"]}

# ── Context Pruner (TTL: settlement window = 4 hours) ────────────
def context_pruner(state: TradeState) -> dict:
    TTL = 14400  # 4 hours — trade signals outside window are irrelevant
    now = time.time()
    active = [s for s in state["signals"] if now - s.get("ts", now) < TTL]
    print(f"  [ContextPruner] {len(active)}/{len(state['signals'])} signals within TTL")
    return {"active_signals":active, "audit_log":[f"[PRUNE] {len(active)} signals retained"]}

# ── Compliance Guardrail — NO LLM. Rules only. Deterministic. ─────
def compliance_guardrail(state: TradeState) -> dict:
    signals = {s["source"]:s for s in state["active_signals"]}
    notes, passed = [], True
    # Rule 1: counterparty must be on approved list
    if not signals.get("counterparty",{}).get("approved",False):
        notes.append("FAIL: Counterparty not on approved list"); passed = False
    # Rule 2: regulatory breach → must quarantine
    if signals.get("regulatory",{}).get("reporting_breach",False):
        notes.append("FAIL: MiFID pre-trade reporting breach"); passed = False
    # Rule 3: liquidity must cover 1.2x
    if not signals.get("liquidity",{}).get("adequate",True):
        notes.append("FAIL: Insufficient liquidity coverage"); passed = False
    status = "PASSED" if passed else f"FAILED ({len(notes)} violations)"
    print(f"  [ComplianceGuardrail] {status}")
    return {"compliance_passed":passed, "compliance_notes":notes,
            "audit_log":[f"[COMPLIANCE] {status}"]}

TypeError: 'NoneType' object is not callable